# 44. The lookup channel, and a protocol error in rows 16 and 33

Two arms, a chain of one-variable steps rather than one comparison:

| arm | one variable against | what changes |
|---|---|---|
| `neural_fixed` | row 33 `neural_te`, 0.965373 | the **epoch-selection rule** |
| `neural_lookup` | `neural_fixed`, measured in this kernel | the **per-value embedding channel** |

## The protocol error, which has to come first

Rows 16 and 33 both train for up to 30 epochs, keep the epoch with the best **validation-fold
AUC**, and then report that fold's AUC as the fold score. The model is selected on the fold it is
scored on. The ledger records the rule plainly, in both rows, as `early stop on fold AUC`. What
neither row records is that **this makes the number optimistic**.

That matters more than usual here, because `35_native_catboost.ipynb` declined to early-stop for
exactly this reason and said so in its header:

> Public notebooks on this competition early-stop on the validation fold, which is the fold their
> OOF is scored on. That is a mild optimism.

So this repo has been holding public notebooks to a standard two of its own members do not meet.
Both are in the stack: row 33 `neural_te` carries +0.0717 in row 94's 41-member combiner and row
16 `neural` carries +0.0038.

**The fix is not a smaller patience.** It is to remove validation from the selection entirely.
`neural_fixed` trains the full 30-epoch OneCycle schedule and takes the **final** model. OneCycle
anneals the learning rate to near zero by design, so the last epoch is the natural stopping point
and no validation signal is consulted at any stage.

`neural_fixed` therefore measures the size of the optimism, which is a number this repo currently
does not have and should.

## The lookup channel

Row 33 embeds only the three categorical columns and feeds the nine numeric columns, their nine
encodings, their nine frequency encodings and a missing mask through a quantile transform as
dense input.

The lookup arm adds **one embedding table per numeric column, keyed on the exact value**, with an
extra row for missing. Vocabulary comes from train and test together, which uses no target
information at all, only the set of values that exist.

| column | vocabulary |
|---|---|
| `weekend_screen_time` | 1,459 |
| `daily_screen_time_hours` | 1,397 |
| `social_media_hours` | 729 |
| the other six | 18 to 601 |

About 5,500 values in total, times 8 dimensions, so roughly 44,000 new parameters against 691,369
rows. Each distinct value is seen about 100 times in a training fold.

**Why this and not something else.** tamerlanomralinov's public work argues this data's target
responds to a value as a **key** rather than as a magnitude: `notifications_per_day` reaches a
univariate AUC of 0.492, no monotone relationship at all, while per-value residuals correlate at
0.72 across independent slices. An embedding table is the natural way to give a model that
channel directly, and it is the one representation no member of this stack has.

## The honest case against, and it is strong

Rows 101 to 105 measured `max_bin` at **+0.000049** and concluded that target encoding and bin
resolution are **substitutes**: both recover the same per-value structure, and doing the first
leaves nothing for the second. A per-value embedding is a third route to the same information, so
the same argument predicts it is largely redundant with the encoder that is already in the input.

If that holds, the lookup arm is null and the useful output of this notebook is the protocol
measurement rather than a new member.

**The counter, and the reason to run it anyway:** an embedding is not a scalar. Target encoding
gives each value one number, its smoothed target rate. An embedding gives each value eight free
parameters that interact with the rest of the network, so it can express things a single number
cannot. Whether that matters is exactly what is unmeasured.

## The prediction, written before the run

- **`neural_fixed` comes in BELOW row 33**, by +0.0003 to +0.0015. That gap is the optimism, and
  the sign is not in doubt: selecting the best of 30 epochs on the scored fold cannot be
  pessimistic. Only the size is a question.
- **`neural_lookup` is between -0.0005 and +0.0025 against `neural_fixed`**, which is a wide band
  and deliberately so. I have now been wrong on three consecutive magnitude predictions, and the
  substitutes argument above and the embedding argument point in opposite directions.

The one thing I will commit to: **whatever `neural_lookup` scores, membership is decided by the
gate and not by its solo CV.** Row 96 added a model 0.0033 weaker than the stack it joined and it
fired at 5/5 folds.

## What this decides

Nothing about the stack. It writes member vectors and one honest number about the two neural rows
already in it. Membership is a separate notebook and a separate ledger row. No submission csv.

In [ ]:
# SMOKE trims everything to a couple of minutes on CPU. Run it once before spending a
# GPU session: it exercises every line on a small subset, so a crash costs two minutes
# instead of a wasted session.
SMOKE = True

SEED = 42
N_SPLITS = 5
N_INNER = 5
SMOOTH = 10.0
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

# Row 33's architecture and schedule, held fixed so the only differences are the two
# named in the header.
EMB_DIM = 8
HIDDEN = (512, 256, 128)
DROPOUT = 0.2
EPOCHS, BATCH, LR, WD = 30, 4096, 3e-3, 1e-4
USE_MISSING_MASK = True

# THE PROTOCOL CHANGE. Row 33 kept the best-validation-AUC epoch, which selects the model
# on the fold it is scored on. Both arms here train the full OneCycle schedule and take
# the FINAL epoch. OneCycle anneals the learning rate to near zero, so the last epoch is
# the natural stopping point and no validation signal is consulted at any stage.
SELECT_ON_VAL = False

# The new channel, and the arm that does not use it.
ARMS = ["neural_fixed", "neural_lookup"]
VALUE_EMB_DIM = 8

ROW33_CV = 0.965373
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"

if SMOKE:
    EPOCHS, N_SPLITS = 2, 2

print(f"SMOKE = {SMOKE}   arms {ARMS}")
print(f"epochs {EPOCHS}, select_on_val {SELECT_ON_VAL} (row 33 used True)")

In [ ]:
import ast
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda" and not SMOKE:
    print("\nWARNING: no GPU. Set the accelerator, or this takes about an hour.")

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
COLS = [c for c in train_full.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

checks = {
    "id is not a feature": ID not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full[ID]) & set(test[ID])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != ID],
}
for k, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {k}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy().astype(np.float32)

# The alignment gate. Computed locally on 2026-08-04 with scikit-learn 1.9.0. If any
# disagree, the out-of-fold vector this notebook produces cannot be blended with the
# local ones and the run is worthless.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

print()
if SMOKE:
    print("  SMOKE subsamples the data, so the fold split cannot match.")
    print("  Alignment is NOT checked and this OOF is unusable.")
ALIGNED = not SMOKE
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")
print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"{len(train):,} train rows, {len(test):,} test rows")

## The encoder, fingerprinted against 13

Copied from `13_target_encoding.ipynb` exactly as rows 17, 19 and 33 did, and checksummed through
`ast.unparse` so a silently different encoder cannot pass as one variable.

In [ ]:
X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")


def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    return (hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]
            if len(parts) == len(ENCODER_FNS) else None)


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))
theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here  : {mine}")
print(f"encoder fingerprint in 13 : {theirs}")
print("encoder: IDENTICAL to rows 17, 26 and 31's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - not comparable to those rows")
ENC_COLS = [f"{p}_{c}" for c in COLS for p in ("te", "fq")]
print(f"\nnumeric block: {len(NUM_COLS)} raw + {len(ENC_COLS)} encoded = "
      f"{len(NUM_COLS) + len(ENC_COLS)} columns, plus {len(NUM_COLS)} mask columns")
print(f"row 16 had {len(NUM_COLS)} numeric + {len(NUM_COLS)} mask")

In [ ]:
# The same three leak checks 13 ran, on the same encoder. Read together: the first two
# must be about zero, the third must be large. Without the third, an encoder that
# ignored the target entirely would pass the first two and look clean.
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

import gc

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## The two input representations

`neural_fixed` uses row 33's inputs unchanged. `neural_lookup` adds the per-value embedding
tables. Both vocabularies are built from train and test together, which uses no target
information, only the set of values that exist.

In [ ]:
# Categorical codes. Fit on train and test together, which is safe: it uses no target
# information whatsoever, only the set of levels that exist. Unchanged from rows 16 and 33.
cat_sizes = []
codes_tr, codes_te = {}, {}
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}      # 0 reserved for missing
    codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)

Xc_tr = np.stack([codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([codes_te[c] for c in CAT_COLS], axis=1)

# THE LOOKUP CHANNEL. One vocabulary per numeric column, keyed on the exact value, with 0
# reserved for missing. Same construction as the categorical codes above and the same leak
# argument: the set of values that exist is not target information.
val_sizes = []
vcodes_tr, vcodes_te = {}, {}
for c in NUM_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True)
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}
    vcodes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    vcodes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    val_sizes.append(len(levels) + 1)

Xv_tr = np.stack([vcodes_tr[c] for c in NUM_COLS], axis=1)
Xv_te = np.stack([vcodes_te[c] for c in NUM_COLS], axis=1)

print(f"{'column':<26}{'vocabulary':>12}{'rows per value':>16}")
for c, s in zip(NUM_COLS, val_sizes):
    print(f"  {c:<24}{s:>12,}{len(train) / max(s - 1, 1):>16,.0f}")
print(f"\nvalue-embedding parameters: {sum(val_sizes) * VALUE_EMB_DIM:,} "
      f"over {len(train):,} rows")

# Every value in test must be inside the vocabulary or it silently becomes 'missing'.
unseen = {c: int((Xv_te[:, i] == 0).sum() - test[c].isna().sum())
          for i, c in enumerate(NUM_COLS)}
print(f"test rows mapped to the missing token beyond genuine NaN: "
      f"{max(unseen.values())} (must be 0)")
VOCAB_OK = max(unseen.values()) == 0

# The mask is taken on the ORIGINAL numeric columns only. The encoded columns have no
# NaN by construction, since _apply sends an unseen level to the prior.
mask_tr = np.isnan(train[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)
mask_te = np.isnan(test[NUM_COLS].to_numpy().astype(np.float32)).astype(np.float32)

In [ ]:
class TabMLP(nn.Module):
    """Row 33's network. `val_sizes=None` reproduces it exactly; passing vocabularies
    adds one embedding table per numeric column keyed on the exact value."""

    def __init__(self, n_num, cat_sizes, val_sizes=None, emb_dim=EMB_DIM,
                 vemb_dim=VALUE_EMB_DIM, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim) for s in cat_sizes])
        self.vembs = nn.ModuleList([nn.Embedding(s, vemb_dim) for s in val_sizes]) \
            if val_sizes else None
        dim = n_num + emb_dim * len(cat_sizes)
        if val_sizes:
            dim += vemb_dim * len(val_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc, xv=None):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        if self.vembs is not None:
            e += [emb(xv[:, i]) for i, emb in enumerate(self.vembs)]
        return self.net(torch.cat([xn] + e, dim=1)).squeeze(1)


NUM_WORKERS = 0 if os.name == "nt" else 2


def make_loader(xn, xc, xv, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc), torch.from_numpy(xv),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(
        ds, batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=(DEV.type == "cuda"), drop_last=drop_last)


@torch.no_grad()
def predict(model, loader, use_v):
    model.eval()
    out = []
    for xn, xc, xv, _ in loader:
        p = model(xn.to(DEV), xc.to(DEV), xv.to(DEV) if use_v else None)
        out.append(torch.sigmoid(p).float().cpu().numpy())
    return np.concatenate(out)


LOG = OUT / "44_neural_lookup.log"


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, device={DEV}, arms={ARMS} ===")

In [ ]:
from sklearn.preprocessing import QuantileTransformer

results = {}
t_start = time.time()

for arm in ARMS:
    use_v = arm.endswith("lookup")
    oof = np.zeros(len(train), dtype=np.float64)
    test_pred = np.zeros(len(test), dtype=np.float64)
    fold_scores, curves = [], []
    t_arm = time.time()

    for f in range(N_SPLITS):
        seed_all(SEED + f)
        tr_i = np.where(folds != f)[0]
        va_i = np.where(folds == f)[0]

        # The encoder runs inside the fold, as in row 33.
        Etr, Eva, Ete = build(X, y, tr_i, va_i, X_test)
        num_tr = Etr[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)
        num_va = Eva[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)
        num_te = Ete[NUM_COLS + ENC_COLS].to_numpy().astype(np.float32)

        med = np.nanmedian(num_tr, axis=0)

        def prep(a):
            return np.where(np.isnan(a), med, a)

        qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                                 subsample=200_000, random_state=SEED)
        qt.fit(prep(num_tr))

        def finish(a, m):
            x = qt.transform(prep(a)).astype(np.float32)
            return np.hstack([x, m]) if USE_MISSING_MASK else x

        Xn_tr = finish(num_tr, mask_tr[tr_i])
        Xn_va = finish(num_va, mask_tr[va_i])
        Xn_te = finish(num_te, mask_te)

        tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], Xv_tr[tr_i], y[tr_i],
                                BATCH, True, drop_last=True)
        va_loader = make_loader(Xn_va, Xc_tr[va_i], Xv_tr[va_i], None, BATCH * 4, False)
        te_loader = make_loader(Xn_te, Xc_te, Xv_te, None, BATCH * 4, False)

        model = TabMLP(Xn_tr.shape[1], cat_sizes,
                       val_sizes if use_v else None).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr_loader))
        lossf = nn.BCEWithLogitsLoss()

        # No best_state, no patience, no validation in the loop. The curve is recorded
        # for the diagnostic below but is never used to choose anything.
        curve = []
        for ep in range(EPOCHS):
            model.train()
            for xn, xc, xv, yy in tr_loader:
                opt.zero_grad(set_to_none=True)
                out = model(xn.to(DEV), xc.to(DEV), xv.to(DEV) if use_v else None)
                loss = lossf(out, yy.to(DEV))
                loss.backward()
                opt.step()
                sched.step()
            if ep % 5 == 0 or ep == EPOCHS - 1:
                a = roc_auc_score(y[va_i], predict(model, va_loader, use_v))
                curve.append((ep, float(a)))
                print(f"  {arm} fold {f} epoch {ep:>2}: val AUC {a:.6f}")

        oof[va_i] = predict(model, va_loader, use_v)
        test_pred += predict(model, te_loader, use_v) / N_SPLITS
        fold_scores.append(float(roc_auc_score(y[va_i], oof[va_i])))
        curves.append(curve)
        el = time.time() - t_start
        note(f"{arm} fold {f}: AUC {fold_scores[-1]:.6f}   elapsed {el/60:.1f} min")

    results[arm] = {"oof": oof, "test": test_pred,
                    "per": np.array(fold_scores), "curves": curves}
    note(f"{arm}: CV {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f} "
         f"in {(time.time()-t_arm)/60:.1f} min")

print(f"\nboth arms done in {(time.time()-t_start)/60:.1f} min")

In [ ]:
def load_saved(stem):
    for name in (f"{stem}_oof.npy", f"{stem}.npy"):
        try:
            return np.load(locate(name))
        except FileNotFoundError:
            continue
    return None


fixed = results["neural_fixed"]["per"]
lookup = results["neural_lookup"]["per"]

print(f"{'arm':16}{'CV':>11}{'sd':>10}")
for a in ARMS:
    p = results[a]["per"]
    print(f"{a:16}{p.mean():11.6f}{p.std():10.6f}")

print(f"\n1. THE OPTIMISM IN ROWS 16 AND 33, which is the number this repo did not have.")
v33 = load_saved("neural_te")
if v33 is not None and not SMOKE:
    p33 = np.array([roc_auc_score(y[folds == f], v33[folds == f])
                    for f in range(N_SPLITS)])
    d = fixed - p33
    sd = d.std(ddof=1)
    print(f"   row 33, selected on the scored fold : {p33.mean():.6f}")
    print(f"   same model, final epoch, no selection: {fixed.mean():.6f}")
    print(f"   optimism: {-d.mean():+.6f}, sd {sd:.6f}, "
          f"{int((d < 0).sum())}/{N_SPLITS} folds worse without selection")
    print(f"   row 33's ledger value {ROW33_CV:.6f}, refit check "
          f"{p33.mean() - ROW33_CV:+.2e}")
else:
    print("   SMOKE or vector missing: not measured.")

print(f"\n2. THE LOOKUP CHANNEL, one variable against neural_fixed in this same kernel.")
d = lookup - fixed
sd = d.std(ddof=1)
t = d.mean() / (sd / np.sqrt(N_SPLITS)) if sd > 0 else float("nan")
print(f"   paired {d.mean():+.6f}, sd {sd:.6f}, {int((d > 0).sum())}/{N_SPLITS} folds, "
      f"t({N_SPLITS - 1})={t:.2f}")
print(f"   per-fold {np.round(d, 6).tolist()}")
print(f"\n   for scale, the same information by other routes:")
print(f"     max_bin 256 -> 1536 on the encoded frame  +0.000049   rows 101 to 105")
print(f"     the ratio block on the encoded frame      +0.000906   row 93")
if SMOKE:
    print("\nSMOKE: subsampled, none of the above resolves a difference this small.")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for arm in ARMS:
    np.save(OUT / f"{pre}{arm}_oof.npy", results[arm]["oof"])
    np.save(OUT / f"{pre}{arm}_test.npy", results[arm]["test"])
    print(f"wrote {pre}{arm}_oof.npy, {pre}{arm}_test.npy")

print("\nledger lines:")
for arm in ARMS:
    p = results[arm]["per"]
    print(f"  name    {arm}\n  cv_mean {p.mean():.6f}\n  cv_std  {p.std():.6f}")
print(f"\n  encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"vocabulary covers test {VOCAB_OK}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is a separate notebook and a separate ledger row.")